In [10]:
import pandas as pd
df=pd.read_csv("cleaned_data.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [11]:
customer_df=df.groupby("CustomerID").size().reset_index(name="purchase_frequency")
customer_df["Totalspending"]=customer_df["CustomerID"].map(df.groupby("CustomerID")["TotalAmount"].sum())
customer_df["Last_perchased"]=customer_df["CustomerID"].map(df.groupby("CustomerID")["InvoiceDate"].max())
customer_df["Recency"]=(df["InvoiceDate"].max()-customer_df["Last_perchased"]).dt.days
customer_df["Churn"]=(customer_df["Recency"]>=90).astype(int)

In [12]:
customer_df.head()
# end_date=df["InvoiceDate"].max()
# print(end_date)
#customer_df["Churn"].value_counts()
#customer_df["Churn"].value_counts(normalize=True)*100
#customer_df.shape

,CustomerID,purchase_frequency,Totalspending,Last_perchased,Recency,Churn
0,12346,1,77183.60,2011-01-18 10:01:00,325,1
1,12347,182,4310.00,2011-12-07 15:52:00,1,0
2,12348,31,1797.24,2011-09-25 13:13:00,74,0
3,12349,73,1757.55,2011-11-21 09:51:00,18,0
4,12350,17,334.40,2011-02-02 16:01:00,309,1


In [ ]:
refrence_date=pd.Timestamp("2011-9-10")
print(refrence_date)
df_before=df[df["InvoiceDate"]<=refrence_date]
df_after=df[df["InvoiceDate"]>refrence_date]
print(df_after.shape)
print(df_before.shape)

2011-09-10 00:00:00
(158852, 9)
(233840, 9)


InvoiceNo               int64
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
TotalAmount           float64
dtype: object

In [14]:
training_df=df_before.groupby("CustomerID").agg(
    purcahse_frequency=("InvoiceDate","nunique"),
    Total_spending=("TotalAmount","sum"),
    last_purchased=("InvoiceDate","max")
).reset_index()
training_df["recency"]=(refrence_date-training_df["last_purchased"]).dt.days

In [15]:
future_purchase=df_after[
    df_after["InvoiceDate"]<=pd.Timestamp("2011-12-9")].groupby("CustomerID").size()

In [16]:
training_df.head()
#future_purchase.head()
#training_df["churn"].value_counts()

,CustomerID,purcahse_frequency,Total_spending,last_purchased,recency
0,12346,1,77183.60,2011-01-18 10:01:00,234
1,12347,5,2790.86,2011-08-02 08:48:00,38
2,12348,3,1487.24,2011-04-05 10:47:00,157
3,12350,1,334.40,2011-02-02 16:01:00,219
4,12352,5,1561.81,2011-03-22 16:08:00,171


In [17]:
next_purcahse=df_after.groupby("CustomerID")["InvoiceDate"].min()
training_df["next_purchase"]=training_df["CustomerID"].map(next_purcahse)
training_df["Gap"]=(
    training_df["next_purchase"]-training_df["last_purchased"]
).dt.days


In [18]:
training_df["churn"]=(
    training_df["Gap"].isna()|(training_df["Gap"]>90)
).astype(int)

In [21]:
training_df["churn"].value_counts(normalize=True)*100
training_df.head()

,CustomerID,purcahse_frequency,Total_spending,last_purchased,recency,next_purchase,Gap,churn
0,12346,1,77183.60,2011-01-18 10:01:00,234,NaT,NaN,1
1,12347,5,2790.86,2011-08-02 08:48:00,38,2011-10-31 12:25:00,90.0,0
2,12348,3,1487.24,2011-04-05 10:47:00,157,2011-09-25 13:13:00,173.0,1
3,12350,1,334.40,2011-02-02 16:01:00,219,NaT,NaN,1
4,12352,5,1561.81,2011-03-22 16:08:00,171,2011-09-20 14:34:00,181.0,1
